# PETadex sample density, normalized by Logan sample density

Same equal-area Mollweide grid as [plot_map.ipynb](plot_map.ipynb), but instead of plotting raw petadex catalytic-ORF sequence counts, this plots, per grid cell:

```
ratio = (# petadex samples in cell) / (# Logan samples in cell)
```

with `ratio = 0` where a cell has 0 Logan samples (avoids division by zero).

**Note on plotting:** cells with 0 Logan samples (mostly ocean/uninhabited area, ~61% of the grid) get `ratio = 0` in the data, same numeric value as cells that *do* have Logan samples but 0 petadex hits. To keep those two cases visually distinguishable, the plot below only colors cells that have at least one Logan sample — cells with zero Logan samples are left transparent, same as the original plots. The underlying `ratio` column still has 0 (not NaN) for the zero-Logan-sample case, per the requested calculation.

In [1]:
import pandas as pd

In [2]:
df_petadex = pd.read_parquet("petadex_library_geo_density.parquet")
df_logan = pd.read_parquet("logan_biosample_geo.parquet")

In [3]:
import os
import urllib.request
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Polygon, box

MOLLWEIDE = "ESRI:54009"
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# ———— WORLD BASEMAP (Natural Earth 1:50m countries, matching R's ne_countries(scale = "medium")) ———— #
ne_shp_path = os.path.join(CACHE_DIR, "ne_50m_admin_0_countries.shp")
if not os.path.exists(ne_shp_path):
    ne_zip_path = os.path.join(CACHE_DIR, "ne_50m_admin_0_countries.zip")
    url = "https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip"
    urllib.request.urlretrieve(url, ne_zip_path)
    with zipfile.ZipFile(ne_zip_path) as z:
        z.extractall(CACHE_DIR)

world = gpd.read_file(ne_shp_path)
world = world.to_crs(MOLLWEIDE)

In [4]:
# ———— EQUAL-AREA GRID (100km cells), MASKED TO THE PROJECTED GLOBE OUTLINE ———— #
# Densify the lon/lat bounding rectangle before projecting, to avoid pole-singularity
# artifacts in Mollweide (mirrors terra::densify + project in the R code).
cell_size = 100_000  # 100km


def densify_rectangle_coords(minx, miny, maxx, maxy, interval=1.0):
    xs = np.arange(minx, maxx, interval)
    ys = np.arange(miny, maxy, interval)
    top = list(zip(xs, np.full_like(xs, maxy))) + [(maxx, maxy)]
    right = list(zip(np.full_like(ys, maxx), ys[::-1])) + [(maxx, miny)]
    bottom = list(zip(xs[::-1], np.full_like(xs, miny))) + [(minx, miny)]
    left = list(zip(np.full_like(ys, minx), ys)) + [(minx, maxy)]
    return top + right + bottom + left


rect_coords = densify_rectangle_coords(-179.9, -89.9, 179.9, 89.9, interval=1.0)
to_moll = Transformer.from_crs("EPSG:4326", MOLLWEIDE, always_xy=True)
proj_coords = [to_moll.transform(x, y) for x, y in rect_coords]
world_ellipse = Polygon(proj_coords)

minx, miny, maxx, maxy = world_ellipse.bounds
nx = int(np.ceil((maxx - minx) / cell_size))
ny = int(np.ceil((maxy - miny) / cell_size))

# Candidate cell centers, vectorized, then keep only cells whose center falls
# inside the projected globe outline (equivalent to terra::mask).
col_idx, row_idx = np.meshgrid(np.arange(nx), np.arange(ny))
col_idx = col_idx.ravel()
row_idx = row_idx.ravel()
cx = minx + (col_idx + 0.5) * cell_size
cy = miny + (row_idx + 0.5) * cell_size

centers = gpd.GeoSeries(gpd.points_from_xy(cx, cy), crs=MOLLWEIDE)
inside_mask = centers.within(world_ellipse).to_numpy()

valid_col = col_idx[inside_mask]
valid_row = row_idx[inside_mask]
grid_ids = np.arange(1, len(valid_col) + 1)

# Lookup table: (row, col) -> id, -1 = masked out / no cell
id_lookup = np.full((ny, nx), -1, dtype=np.int64)
id_lookup[valid_row, valid_col] = grid_ids

cells = [
    box(minx + c * cell_size, miny + r * cell_size, minx + (c + 1) * cell_size, miny + (r + 1) * cell_size)
    for c, r in zip(valid_col, valid_row)
]
grid_sf = gpd.GeoDataFrame({"id": grid_ids}, geometry=cells, crs=MOLLWEIDE)


def assign_grid_id(lon, lat):
    """Vectorized raster-index lookup, equivalent to a point-in-polygon join (st_intersects)
    against this axis-aligned equal-area grid."""
    x, y = to_moll.transform(lon, lat)
    col = np.floor((x - minx) / cell_size).astype(np.int64)
    row = np.floor((y - miny) / cell_size).astype(np.int64)
    in_bounds = (col >= 0) & (col < nx) & (row >= 0) & (row < ny)
    gid = np.full(len(lon), -1, dtype=np.int64)
    gid[in_bounds] = id_lookup[row[in_bounds], col[in_bounds]]
    return gid

In [5]:
# ———— COUNT PETADEX SAMPLES AND LOGAN SAMPLES PER GRID CELL, THEN TAKE THE RATIO ———— #
df_petadex["grid_id"] = assign_grid_id(df_petadex["longitude"].to_numpy(), df_petadex["latitude"].to_numpy())
df_logan["grid_id"] = assign_grid_id(df_logan["longitude"].to_numpy(), df_logan["latitude"].to_numpy())

petadex_count = df_petadex[df_petadex["grid_id"] > 0].groupby("grid_id").size().rename("petadex_count")
logan_count = df_logan[df_logan["grid_id"] > 0].groupby("grid_id").size().rename("logan_count")

grid_sf_density = grid_sf.merge(petadex_count, left_on="id", right_index=True, how="left")
grid_sf_density = grid_sf_density.merge(logan_count, left_on="id", right_index=True, how="left")
grid_sf_density["petadex_count"] = grid_sf_density["petadex_count"].fillna(0)
grid_sf_density["logan_count"] = grid_sf_density["logan_count"].fillna(0)

# Ratio of petadex samples to Logan samples per cell; 0 where there are 0 Logan samples.
grid_sf_density["ratio"] = np.where(
    grid_sf_density["logan_count"] > 0,
    grid_sf_density["petadex_count"] / grid_sf_density["logan_count"],
    0.0,
)

In [ ]:
# ———— PLOT: WORLD MAP COLOURED BY PETADEX / LOGAN SAMPLE RATIO ———— #
cm_to_in = 1 / 2.54
fig, ax = plt.subplots(figsize=(12.5 * cm_to_in, 6 * cm_to_in), dpi=300)

world.plot(ax=ax, facecolor="#e6e6e6", edgecolor="#d8d8d8", linewidth=0.3, zorder=1)

cmap = mcolors.LinearSegmentedColormap.from_list("density", ["#f2b3b4", "#dd3c3c", "#3f0b0b"])

# Only plot cells that actually have Logan samples, so uninhabited/unsampled area (no data)
# stays transparent instead of being painted the same "0" color as a real zero ratio.
plot_data = grid_sf_density[grid_sf_density["logan_count"] > 0]

# The ratio is heavily right-skewed (a handful of cells run up to ~6x due to grid-boundary
# discretization noise), so cap the color scale at 1 -- the 99th percentile of nonzero ratios --
# to keep the typical 0-1 range legible; higher cells just saturate at the darkest color.
norm = mcolors.Normalize(vmin=0, vmax=1)
plot_data.plot(
    ax=ax,
    column="ratio",
    cmap=cmap,
    norm=norm,
    edgecolor="none",
    legend=True,
    legend_kwds={"label": "petadex / Logan samples", "shrink": 0.6, "extend": "max"},
    zorder=2,
)

ax.set_axis_off()
ax.set_facecolor("#f5f5f5")
fig.patch.set_alpha(0)

os.makedirs("outputs", exist_ok=True)
fig.savefig("outputs/petadex_logan_ratio_moll.png", dpi=300, transparent=True, bbox_inches="tight")
fig.savefig("outputs/petadex_logan_ratio_moll.svg", transparent=True, bbox_inches="tight")

plt.show()